In [1]:
from pymongo import MongoClient
uri = "mongodb://localhost:27017/"
def execute(callable):
    try:
        client = MongoClient(uri)
        callable(client)

        client.close()
    except Exception as e:
        raise Exception("Unable to find the document due to the following error: ", e)

def find(cursor):
    for c in cursor:
        print(c)

In [3]:
from datetime import datetime
# 1. STUDENTS Collection - For grade calculations
students_data = [
    {"name": "Alice Johnson", "math": 85, "science": 92, "english": 78, "credits": 18},
    {"name": "Bob Smith", "math": 72, "science": 68, "english": 85, "credits": 15},
    {"name": "Carol Davis", "math": 95, "science": 88, "english": 92, "credits": 21},
    {"name": "David Wilson", "math": 67, "science": 74, "english": 81, "credits": 12},
    {"name": "Emma Brown", "math": 89, "science": 91, "english": 87, "credits": 20},
    {"name": "Frank Miller", "math": 76, "science": 82, "english": 79, "credits": 16},
    {"name": "Grace Lee", "math": 93, "science": 96, "english": 94, "credits": 22},
    {"name": "Henry Taylor", "math": 81, "science": 75, "english": 83, "credits": 18}
]

# 2. PRODUCTS Collection - For price/profit calculations
products_data = [
    {"name": "Laptop", "costPrice": 800, "sellingPrice": 1200, "quantity": 50},
    {"name": "Mouse", "costPrice": 15, "sellingPrice": 25, "quantity": 200},
    {"name": "Keyboard", "costPrice": 45, "sellingPrice": 75, "quantity": 120},
    {"name": "Monitor", "costPrice": 250, "sellingPrice": 400, "quantity": 80},
    {"name": "Headphones", "costPrice": 60, "sellingPrice": 99, "quantity": 150},
    {"name": "Webcam", "costPrice": 35, "sellingPrice": 65, "quantity": 90},
    {"name": "Speaker", "costPrice": 80, "sellingPrice": 130, "quantity": 60},
    {"name": "Tablet", "costPrice": 300, "sellingPrice": 450, "quantity": 40}
]

# 3. SALES Collection - For revenue calculations
sales_data = [
    {"product": "Laptop", "quantity": 5, "unitPrice": 1200, "date": datetime(2024, 1, 15)},
    {"product": "Mouse", "quantity": 20, "unitPrice": 25, "date": datetime(2024, 1, 16)},
    {"product": "Keyboard", "quantity": 8, "unitPrice": 75, "date": datetime(2024, 1, 17)},
    {"product": "Monitor", "quantity": 3, "unitPrice": 400, "date": datetime(2024, 1, 18)},
    {"product": "Headphones", "quantity": 12, "unitPrice": 99, "date": datetime(2024, 1, 19)},
    {"product": "Laptop", "quantity": 2, "unitPrice": 1200, "date": datetime(2024, 1, 20)},
    {"product": "Tablet", "quantity": 6, "unitPrice": 450, "date": datetime(2024, 1, 21)},
    {"product": "Speaker", "quantity": 4, "unitPrice": 130, "date": datetime(2024, 1, 22)}
]

# 4. EMPLOYEES Collection - For salary calculations
employees_data = [
    {"name": "John Manager", "baseSalary": 5000, "bonus": 1000, "hoursWorked": 160, "hourlyRate": 0},
    {"name": "Jane Developer", "baseSalary": 4000, "bonus": 800, "hoursWorked": 170, "hourlyRate": 0},
    {"name": "Mike Contractor", "baseSalary": 0, "bonus": 0, "hoursWorked": 120, "hourlyRate": 50},
    {"name": "Sarah Designer", "baseSalary": 3500, "bonus": 600, "hoursWorked": 160, "hourlyRate": 0},
    {"name": "Tom Freelancer", "baseSalary": 0, "bonus": 200, "hoursWorked": 80, "hourlyRate": 75},
    {"name": "Lisa Analyst", "baseSalary": 3800, "bonus": 500, "hoursWorked": 165, "hourlyRate": 0}
]

execute(lambda client: client.exam.students.insert_many(students_data))
execute(lambda client: client.exam.products.insert_many(products_data))
execute(lambda client: client.exam.sales.insert_many(sales_data))
execute(lambda client: client.exam.employees.insert_many(employees_data))

6. Count how many products are in each product line

- First solution

In [ ]:
[
    {
        '$lookup': {
            'from': 'products', 
            'localField': 'productLine', 
            'foreignField': 'product.Line', 
            'as': 'productlines_products'
        }
    }, {
        '$addFields': {
            'count_products': {
                '$size': [
                    '$productlines_products'
                ]
            }
        }
    }, {
        '$project': {
            'productLine': 1, 
            'count_products': 1, 
            '_id': 0
        }
    }
]

- A better solution

In [ ]:
[
    {
        '$group': {
            '_id': '$product.Line', 
            'products_size': {
                '$sum': 1
            }
        }
    }
]

7.	Calculate the total value of all products in stock (quantityInStock × buyPrice) 

In [ ]:
[
    {
        '$group': {
            '_id': None, 
            'total_value': {
                '$sum': {
                    '$multiply': [
                        '$quantityInStock', '$buyPrice'
                    ]
                }
            }
        }
    }
]

- Total value grouped by product line

In [ ]:
[
    {
        '$group': {
            '_id': '$product.Line', 
            'total_value': {
                '$sum': {
                    '$multiply': [
                        '$quantityInStock', '$buyPrice'
                    ]
                }
            }
        }
    }
]

8.	Find the top 5 most expensive products by MSRP

In [ ]:
[
    {
        '$sort': {
            'msrp': -1
        }
    }, {
        '$limit': 5
    }
]

9.	Join employees with offices to show office city and country for each employee

In [ ]:
[
    {
        '$lookup': {
            'from': 'offices', 
            'localField': 'officeCode', 
            'foreignField': 'officeCode', 
            'as': 'employee_office'
        }
    }, {
        '$unwind': {
            'path': '$employee_office'
        }
    }, {
        '$addFields': {
            'city': '$employee_office.city', 
            'country': '$employee_office.country'
        }
    }, {
        '$project': {
            '_id': 0, 
            'firstName': 1, 
            'lastName': 1, 
            'city': 1, 
            'country': 1
        }
    }
]

10.	Show which sales representatives have the highest total sales

In [ ]:
[
    {
        '$lookup': {
            'from': 'customers', 
            'localField': 'employeeNumber', 
            'foreignField': 'customer.salesRepEmployeeNumber', 
            'as': 'employee_customer'
        }
    }, {
        '$unwind': {
            'path': '$employee_customer'
        }
    }, {
        '$lookup': {
            'from': 'orders', 
            'localField': 'employee_customer.customer.number', 
            'foreignField': 'customerNumber', 
            'as': 'customer_order'
        }
    }, {
        '$unwind': {
            'path': '$customer_order'
        }
    }, {
        '$lookup': {
            'from': 'orderdetails', 
            'localField': 'customer_order.orderNumber', 
            'foreignField': 'orderNumber', 
            'as': 'order_details'
        }
    }, {
        '$unwind': {
            'path': '$order_details'
        }
    }, {
        '$group': {
            '_id': '$employeeNumber', 
            'total': {
                '$sum': {
                    '$multiply': [
                        '$order_details.priceEach', '$order_details.quantityOrdered'
                    ]
                }
            }
        }
    }, {
        '$sort': {
            'total': -1
        }
    }
]